# Sample OTTO sessions

Issue: #4. Runs on Kaggle with the `otto/recsys-dataset` dataset attached (CC BY 4.0, see `_doc/decisions/D1-otto-license.md`) - not the competition, which carries separate rules. Locally it can be pointed at a small fake file with the `OTTO_IN` and `OTTO_OUT` environment variables.

## Question

Can we take a reproducible sample of 1 to 2 million whole sessions that fits on an 8GB Mac?

## Data used

Input: `train.jsonl` (four labeled weeks, one session per line). The public test week has hidden labels and is not used.

Output: `events.parquet` (columns `session`, `aid`, `ts`, `type`) and `sample_manifest.json`, downloaded to `data/sample/`.

`type` is 0 = click, 1 = cart, 2 = order. `ts` is Unix milliseconds.

## Why whole sessions

The models learn from session context (what came before the next item). Sampling random *events* would cut sessions into fragments, so prefixes and next items would no longer be real. Sampling whole *sessions* keeps every session intact. The choice of sessions uses a hash of the session ID, not a random generator, so the same sessions are selected on every run and on every machine.

In [ ]:
import hashlib
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq


def find_input() -> Path:
    """OTTO_IN if set, otherwise search the Kaggle input folder for *train.jsonl."""
    if "OTTO_IN" in os.environ:
        return Path(os.environ["OTTO_IN"])
    hits = sorted(Path("/kaggle/input").rglob("*train.jsonl"))
    if not hits:
        listing = [str(p) for p in Path("/kaggle/input").rglob("*")][:20]
        raise FileNotFoundError(f"*train.jsonl not found under /kaggle/input, saw: {listing}")
    return hits[0]


IN_PATH = find_input()
print("input:", IN_PATH)
OUT_DIR = Path(os.environ.get("OTTO_OUT", "/kaggle/working"))
KEEP_PCT = int(os.environ.get("OTTO_KEEP_PCT", "12"))  # keep about 12 percent of sessions
TYPE_CODE = {"clicks": 0, "carts": 1, "orders": 2}
OUT_DIR.mkdir(parents=True, exist_ok=True)


def keep_session(session_id: int, pct: int = KEEP_PCT) -> bool:
    """Stable hash rule. Independent of Python's per-process hash randomisation."""
    h = (session_id * 2654435761) % 2**32
    return (h >> 8) % 100 < pct


SESSION_RE = re.compile(rb'^\{"session":\s*(\d+)')

In [ ]:
def sample(in_path: Path, out_path: Path):
    """Read train.jsonl line by line and keep the whole line of each selected session."""
    sessions, aids, tss, types, seqs = [], [], [], [], []
    counts = {}
    n_seen = 0
    with open(in_path, "rb") as f:
        for line in f:
            n_seen += 1
            sid = int(SESSION_RE.match(line).group(1))
            if not keep_session(sid):
                continue
            events = json.loads(line)["events"]
            counts[sid] = len(events)
            for e in events:
                sessions.append(sid)
                aids.append(e["aid"])
                tss.append(e["ts"])
                types.append(TYPE_CODE[e["type"]])
            seqs.extend(range(len(events)))
    table = pa.table(
        {
            "session": pa.array(sessions, pa.int64()),
            "aid": pa.array(aids, pa.int32()),
            "ts": pa.array(tss, pa.int64()),
            "type": pa.array(types, pa.int8()),
            "seq": pa.array(seqs, pa.int32()),
        }
    )
    order = pc.sort_indices(
        table, sort_keys=[("session", "ascending"), ("ts", "ascending"), ("seq", "ascending")]
    )
    table = table.take(order).drop_columns(["seq"])
    pq.write_table(table, out_path, compression="zstd", row_group_size=1_000_000)
    return table, counts, n_seen


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

In [ ]:
events_path = OUT_DIR / "events.parquet"
table, counts, n_seen = sample(IN_PATH, events_path)
print(f"read {n_seen:,} sessions, kept {len(counts):,} ({len(counts) / n_seen:.1%}), {table.num_rows:,} events")

## Checks

In [ ]:
# 1. every kept session has all of its original events (100 random sessions, compared with the source line)
per_session = table.group_by("session").aggregate([("aid", "count")])
got = dict(zip(per_session["session"].to_pylist(), per_session["aid_count"].to_pylist()))
assert got == counts, "event counts per session differ from the source"
check = random.Random(0).sample(sorted(counts), min(100, len(counts)))
assert all(got[s] == counts[s] for s in check)
print("whole sessions kept: ok")

# 2. sorted by session and time
s = table["session"].to_numpy()
t = table["ts"].to_numpy()
assert ((s[1:] > s[:-1]) | ((s[1:] == s[:-1]) & (t[1:] >= t[:-1]))).all()
print("sorted by session and ts: ok")

# 3. time range stays inside four weeks
days = (int(t.max()) - int(t.min())) / 86_400_000
assert days <= 29, days
print(f"time range: {days:.2f} days")

In [ ]:
# 4. reproducibility: sample again and compare checksums
first = sha256(events_path)
second_path = OUT_DIR / "events_check.parquet"
sample(IN_PATH, second_path)
second = sha256(second_path)
second_path.unlink()
assert first == second, (first, second)
print("same checksum on a second run:", first)

In [ ]:
type_counts = Counter(table["type"].to_pylist())
manifest = {
    "rule": "keep session if ((session * 2654435761) % 2**32 >> 8) % 100 < keep_pct",
    "keep_pct": KEEP_PCT,
    "source": IN_PATH.name,
    "source_sessions": n_seen,
    "sessions": len(counts),
    "events": table.num_rows,
    "events_by_type": {"click": type_counts[0], "cart": type_counts[1], "order": type_counts[2]},
    "ts_min": int(t.min()),
    "ts_max": int(t.max()),
    "days": round(days, 3),
    "sha256": {"events.parquet": first},
}
(OUT_DIR / "sample_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

## Result

The manifest above. The sample is kept in `data/sample/` (gitignored, see `docs/otto-license.md`).

## What I learned

To be written after looking at the real numbers.